In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split                     

#----------------------------------------------------------------------------------  DL모델 
from tensorflow.keras import Sequential, layers, models
from tensorflow.keras.layers import Conv2D, MaxPooling2D, GlobalAveragePooling2D
from tensorflow.keras.layers import Flatten, Dense, Rescaling
from tensorflow.keras.initializers import GlorotNormal    #Xavier

from tensorflow.keras.applications import MobileNetV2, MobileNetV3Small #최소 이미지 크기 (32*32)

#----------------------------------------------------------------------------------  랜덤시드 고정 
import tensorflow as tf
tf.random.set_seed(54546)
np.random.seed(54546)

#----------------------------------------------------------------------------------  EDA : 이미지로드
import os
from tensorflow.keras.utils import load_img, img_to_array
from PIL import Image
from tensorflow.keras.utils import image_dataset_from_directory

#----------------------------------------------------------------------------------  조기종료
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
    

import warnings
warnings.filterwarnings('ignore')

sns.set()

#-------------------- 차트 관련 속성 (한글처리, 그리드) -----------
plt.rcParams['font.family']= 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

#-------------------- 주피터 , 출력결과 넓이 늘리기 ---------------
# from IPython.core.display import display, HTML
from IPython.display import display, HTML
display(HTML("<style>.container{width:100% !important;}</style>"))
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('max_colwidth', None)

# <b>Data Load

In [2]:
import os
os.chdir('C:\\IT\\workspace_python\\dl')

In [3]:
os.getcwd()

'C:\\IT\\workspace_python\\dl'

In [4]:
train = pd.read_csv('./dataset/kannada-mnist/train.csv')
test = pd.read_csv('./dataset/kannada-mnist/test.csv')


In [5]:
train.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,pixel10,pixel11,pixel12,pixel13,pixel14,pixel15,pixel16,pixel17,pixel18,pixel19,pixel20,pixel21,pixel22,pixel23,pixel24,pixel25,pixel26,pixel27,pixel28,pixel29,pixel30,pixel31,pixel32,pixel33,pixel34,pixel35,pixel36,pixel37,pixel38,pixel39,pixel40,pixel41,pixel42,pixel43,pixel44,pixel45,pixel46,pixel47,pixel48,...,pixel734,pixel735,pixel736,pixel737,pixel738,pixel739,pixel740,pixel741,pixel742,pixel743,pixel744,pixel745,pixel746,pixel747,pixel748,pixel749,pixel750,pixel751,pixel752,pixel753,pixel754,pixel755,pixel756,pixel757,pixel758,pixel759,pixel760,pixel761,pixel762,pixel763,pixel764,pixel765,pixel766,pixel767,pixel768,pixel769,pixel770,pixel771,pixel772,pixel773,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
2,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
3,3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,4,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
x_train = train.drop('label', axis=1)
y_train = train['label']

x_test = test.drop('id', axis=1)
y_test = test['id']

In [7]:
x_train = x_train.to_numpy()
y_train = y_train.to_numpy()
x_test = x_test.to_numpy()
y_test = y_test.to_numpy()

In [8]:
from sklearn.model_selection import train_test_split

# X: 특성(Features), y: 타겟(Target)
x_train, x_val, y_train, y_val = train_test_split(
    x_train, y_train, 
    test_size=0.2,       # 테스트 셋 비율 (20%)
    random_state=42,     # 결과 재현을 위한 난수 고정 (필수)
    shuffle=True,        # 데이터 섞기 (시계열 데이터가 아닐 경우)
    stratify=y_train           # 타겟 값의 비율을 유지하며 분할 (매우 중요)
)

In [9]:
x_train.shape, x_val.shape, y_train.shape, y_val.shape

((48000, 784), (12000, 784), (48000,), (12000,))

# 모델

# <font color=red><b>조기종료
* https://keras.io/api/callbacks/

In [10]:
MY_CHECK_POINT = ModelCheckpoint(
                        "./mnist_{epoch}_{val_accuracy:.4f}_{val_loss:.4f}.keras",          #모델저장경로
                        monitor="val_loss",              #체크할점수
                        save_best_only=True,             #점수가 좋아지면 저장
                        save_weights_only=False          #모델+가중치 같이 저장
                    )    

In [11]:
MY_EARLY_STOP = EarlyStopping(
    monitor="val_loss",     #체크할점수
    patience=5              #3회연속 점수가 개선되지 않으면 멈추기
)             

# <font color=red><b>전이학습
* <b>ILSVRC(ImageNet Large Scale Visual Recognition Challenge)</b>
    - Train : 약 128만 장
    - Validation : 5만 장
    - Test : 10만 장
    - 총 클래스 : 1000개
    - https://www.image-net.org/challenges/LSVRC/
* <b>전이학습</b>
    - 잘 알려진 모델을 가져와 내 데이터에 맞게 가중치를 보정해서 재학습하는 것
    - 적은 데이터셋 학습에 효율적
    - https://keras.io/api/applications/mobilenet/mobilenet_models/#mobilenetv2-function

In [12]:
mobile_model = MobileNetV3Small(
    input_shape=(32, 32, 3),
    include_top=False
)

In [13]:
model = Sequential([
    layers.Input(shape=(784,)),             # 1차원 벡터를 입력으로 받음
    layers.Reshape((28, 28, 1)),            # 여기서 이미지 형태로 복원
    layers.Resizing(32, 32),                # 이제 Resizing이 정상 작동함
    Rescaling(1.0/255.0) ,              #---------전처리가공(스케일링)

    # 축(axis) -1을 기준으로 자기 자신을 3번 붙입니다.
    layers.Conv2D(3, (1, 1), padding='same'),

    
    mobile_model                                              ,  #-------- MobileNetV2
    
    Flatten() ,                                                    #FC
    Dense(units=256, activation="relu"                ),           #----- hidden layer 
    Dense(units=10,  activation="softmax"              )            #----- output layer   
 ]) 

In [14]:
# 전체 동결
mobile_model.trainable = False

# 이 상태에서 compile 후 약 5~10 epoch 정도 학습

In [15]:
            
model.compile(loss="sparse_categorical_crossentropy" , optimizer="adam",  metrics=["accuracy"])   #-------------------- l o m

fit_res = model.fit(x_train,y_train, epochs=50, validation_data=(x_val,y_val)
                   , callbacks=[MY_CHECK_POINT, MY_EARLY_STOP])
loss,acc = model.evaluate(x_val,y_val)
print(f"acc : {acc:.4f} , loss:{loss:.4f} " )


Epoch 1/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 41s 25ms/step - accuracy: 0.4713 - loss: 1.4897 - val_accuracy: 0.7203 - val_loss: 0.7952
Epoch 2/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 37s 24ms/step - accuracy: 0.7829 - loss: 0.6453 - val_accuracy: 0.8345 - val_loss: 0.4970
Epoch 3/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 37s 25ms/step - accuracy: 0.8516 - loss: 0.4575 - val_accuracy: 0.8752 - val_loss: 0.3818
Epoch 4/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 38s 25ms/step - accuracy: 0.8779 - loss: 0.3790 - val_accuracy: 0.8997 - val_loss: 0.3127
Epoch 5/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 37s 25ms/step - accuracy: 0.8917 - loss: 0.3365 - val_accuracy: 0.9127 - val_loss: 0.2755
Epoch 6/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 38s 25ms/step - accuracy: 0.9008 - loss: 0.3086 - val_accuracy: 0.9181 - val_loss: 0.2535
Epoch 7/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 38s 25ms/step - accuracy: 0.9069 - loss: 0.2886 - val_accuracy: 0.9218 - val_loss: 0.2412
Epoch 8/50
1500/1500 ━━━━━━━━━━━━━━━━━━━━ 37s 25ms/step - accuracy: 0.9118 -

In [16]:
# # 140이라는 숫자에 집착하지 말고 마지막 블록만 타겟팅하세요
# mobile_model.trainable = True
# for layer in mobile_model.layers[:-30]: # 뒤에서 30개 정도만 해제
#     layer.trainable = False

# # 아주 낮은 학습률 설정 (필수!)
# model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), # 0.00001
#               loss='sparse_categorical_crossentropy',
#               metrics=['accuracy'])


In [17]:
    
# fit_res = model.fit(x_train,y_train, epochs=40, validation_data=(x_val,y_val)
#                    , callbacks=[MY_CHECK_POINT, MY_EARLY_STOP])
# loss,acc = model.evaluate(x_val,y_val)
# print(f"acc : {acc:.4f} , loss:{loss:.4f} " )

In [18]:
from tensorflow.keras.models import load_model
best_model = load_model()
best_model.summary()

TypeError: load_model() missing 1 required positional argument: 'filepath'

In [ ]:
test_dir = image_dataset_from_directory("./dataset/catdog/test"
                            , image_size=(150,150)
                            , seed=4894
                            , batch_size=32
                            , label_mode = None
                            )
proba = best_model.predict(test_dir)
pred = (proba >0.5).astype(int)

In [ ]:
print(pred.shape, pred[:5])
pred = pred.reshape(-1)
print( pred.shape, pred[:5])